In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import chromadb

client = chromadb.Client()  # in-memory; use PersistentClient(path="./chroma_db") to persist across sessions

collection = client.get_or_create_collection(name="faq_docs")
print("Collection ready:", collection.name)

Collection ready: faq_docs


In [3]:
docs = [
    "You can buy a subscription directly from the pricing page using a credit card.",
    "Refunds are processed within 5-7 business days after a cancellation request.",
    "Our support team is available 24/7 via live chat and email.",
    "To reset your password, click 'Forgot Password' on the login screen.",
    "We offer a 14-day free trial with full access to all premium features.",
    "Enterprise plans include dedicated account managers and custom SLAs.",
    "You can cancel your plan anytime from the account settings page.",
    "Two-factor authentication can be enabled under Security Settings.",
    "Our API rate limit is 1000 requests per minute on the Pro plan.",
    "Invoices are automatically emailed at the start of each billing cycle.",
    "Team members can be invited via email from the Workspace settings.",
    "Data is encrypted at rest using AES-256 encryption.",
    "We support integrations with Slack, Zapier, and Google Workspace.",
    "Mobile apps are available for both iOS and Android devices.",
    "Storage limits vary by plan, ranging from 5GB to unlimited.",
    "You can export your data anytime as a CSV or JSON file.",
    "Our uptime SLA guarantees 99.9% availability for Enterprise customers.",
    "Custom domains can be configured under Domain Settings.",
    "We do not store your credit card number on our servers.",
    "Notifications can be customized in the Preferences menu.",
]

# Stable, deterministic IDs so re-running this cell never creates duplicates.
ids = [f"doc_{i:03d}" for i in range(len(docs))]

# Metadata: a "source" field for each doc, for citations later.
metadatas = [{"source": f"faq_{i:03d}.txt"} for i in range(len(docs))]

In [4]:
# upsert = insert if new, overwrite if the ID already exists.
# This is what makes the script safe to re-run without duplicates.
collection.upsert(
    ids=ids,
    documents=docs,
    metadatas=metadatas,
)

print(f"Collection now has {collection.count()} docs (should stay {len(docs)} even after re-running this cell).")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 77.9MiB/s]


Collection now has 20 docs (should stay 20 even after re-running this cell).


In [5]:
query = "How do I purchase a plan?"

results = collection.query(
    query_texts=[query],
    n_results=3,
)

print("=== Chroma Semantic Search ===")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance={dist:.3f}  source={meta['source']}  text={doc}")

=== Chroma Semantic Search ===
distance=0.987  source=faq_000.txt  text=You can buy a subscription directly from the pricing page using a credit card.
distance=1.040  source=faq_006.txt  text=You can cancel your plan anytime from the account settings page.
distance=1.239  source=faq_005.txt  text=Enterprise plans include dedicated account managers and custom SLAs.
